In [6]:
import pandas as pd
import datetime
import os
import json
import altair as alt

df = pd.read_csv("Gesamtdatensatz.csv")

image_paths = ["src/assets/clear-day.png","src/assets/clear-night.png", "src/assets/cloudy.png", "src/assets/fog.png", "src/assets/partly-cloudy-day.png", "src/assets/partly-cloudy-night.png", "src/assets/rain.png", "src/assets/snow.png"]
path = {}

for image_path in image_paths:
    # Schlüssel aus Dateiname extrahieren (ohne Ordner und ohne .png)
    key = os.path.splitext(os.path.basename(image_path))[0]
    # Pfad in das Dictionary einfügen
    path[key] = image_path



# Add columns to table
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['pedestrian_grey'] = df[['ltr_pedestrians_count', 'rtl_pedestrians_count']].min(axis=1)
df['pedestrian_diff'] = ((df['ltr_pedestrians_count'] - df['rtl_pedestrians_count'])**2)**0.5
df['weather_icon'] = df['weather_condition'].map(path)
df['max_val'] = (
    df.groupby(['date'])[['ltr_pedestrians_count', 'rtl_pedestrians_count']]
      .transform('max')      # max per column per date
      .max(axis=1)           # max across the two columns
)+20

f_time_loc = df[(df['timestamp'].dt.date == datetime.date(2021, 10, 18)) & (df['location_name'] == 'Bahnhofstrasse (Mitte)')]
f_time_loc.to_json('time_loc_data.json', orient='records', indent=2)
#f_time = df[(df['timestamp'].dt.date == datetime.date(2021, 10, 15))]
#f_time.to_json('time_data.json', orient='records', indent=2)
#df.to_json('fulldata.json', orient='records', indent=2)

f_time_loc.head(n=5)

,timestamp,location_id,location_name,ltr_label,rtl_label,weather_condition,temperature,pedestrians_count,unverified,collection_type,...,zone_99_ltr_pedestrians_count,zone_99_rtl_pedestrians_count,zone_99_adult_pedestrians_count,zone_99_child_pedestrians_count,date,hour,pedestrian_grey,pedestrian_diff,weather_icon,max_val
1832,2021-10-18 00:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,cloudy,6.06,8,False,measured,...,NaN,NaN,NaN,NaN,2021-10-18,0,2,4.0,src/assets/cloudy.png,1887
1836,2021-10-18 01:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,cloudy,6.00,5,False,measured,...,NaN,NaN,NaN,NaN,2021-10-18,1,1,3.0,src/assets/cloudy.png,1887
1840,2021-10-18 02:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,cloudy,6.10,30,False,measured,...,NaN,NaN,NaN,NaN,2021-10-18,2,6,18.0,src/assets/cloudy.png,1887
1844,2021-10-18 03:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,cloudy,5.78,110,False,measured,...,NaN,NaN,NaN,NaN,2021-10-18,3,41,28.0,src/assets/cloudy.png,1887
1848,2021-10-18 04:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,partly-cloudy-night,5.39,323,False,measured,...,NaN,NaN,NaN,NaN,2021-10-18,4,112,99.0,src/assets/partly-cloudy-night.png,1887


In [ ]:


#data = pd.read_csv("Gesamtdatensatz.csv")
#data = pd.read_json("time_loc_data.json")
data = 'time_loc_data.json'

# Initialize Basic Chart
base = alt.Chart(data).add_params().properties(width=400, height=24*30)

# Build Temperature chart for Background
inv = base.encode(
    alt.Y('hour:T').axis(None),
    alt.X('max_val:Q'), 
    ).mark_rect(
        opacity=0,
        height=0)


temp_left = base.encode(
    alt.Y('hour:T').axis(None), 
    alt.Color('temperature:Q')
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('temperature:Q', 
            title='Temperatur [°C]:', 
            format=".1f")]).mark_rect(
        height=30).interactive()

temp_right = base.encode(
    alt.Y('hour:O').axis(None),
    alt.Color('temperature:Q')
        #.bin(maxbins=10, extent=[-10,35])
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('temperature:Q',
            title='Temperatur [°C]:',
            format=".1f")]).mark_rect(
            height=30).interactive()

# Build Pedestrian-count chart
left = base.encode(
    alt.Y('hour:O').axis(None), 
    alt.X('ltr_pedestrians_count:Q'),
    tooltip=[alt.Tooltip('pedestrian_diff:Q',
        title='Passantendifferenz links zu rechts')]).mark_bar(
                color='darkred',
                height=20).interactive()

right = base.encode(
    alt.Y('hour:O').axis(None), 
    alt.X('rtl_pedestrians_count:Q'),
    tooltip=[alt.Tooltip('pedestrian_diff:Q',
        title='Passantendifferenz rechts zu links')]).mark_bar(
            color='darkred', 
            height=20).interactive()

# Build Pedestrian-even count chart
left_g = base.encode(
    alt.Y('hour:O').axis(None),
    alt.X('sum(pedestrian_grey):Q')
        .sort('descending'),
    tooltip=[alt.Tooltip('ltr_pedestrians_count:Q',
        title=f'Anzahl Passanten')]).mark_bar(
            color='dimgray',
            height=20)

right_g = (
    base
    .transform_calculate(
        tooltip_title="'Anzahl Passanten in Richtung ' + datum.rtl_label"
    )
    .encode(
        alt.Y('hour:O').axis(None),
        alt.X('sum(pedestrian_grey):Q'),
        tooltip=[
            alt.Tooltip('rtl_pedestrians_count:Q'),
            alt.Tooltip('tooltip_title:N', title='')
        ]
    )
    .mark_bar(color='dimgray', height=20)
)

# weather icon chart
weather = base.transform_aggregate(
    weather_icon='max(weather_icon)',
    groupby=['hour', 'location_id']
).encode(
    alt.Y('hour:O').axis(None),
    url='weather_icon:N'
).mark_image(width=25, height=25).properties(width=30)

# Build middle chart (legend)
middle = base.transform_aggregate(groupby=['hour', 'location_id']).encode(
    alt.Y('hour:O').axis(None),
    alt.Text('hour:O')).mark_text(
        fontSize=15,
        font='Bahnschrift').properties(width=20)


# Layer Charts
left_chart = alt.layer(temp_left, left, left_g, inv)
right_chart = alt.layer(temp_right, right, right_g, inv)

# Concatenate Charts
main_chart = alt.concat(weather, left_chart, middle, right_chart, spacing = 5,).configure_view(
    stroke=None,
).configure_legend(direction='horizontal', labelAlign='center')

spec = main_chart.to_dict()

with open("chart.json", "w") as f:
    json.dump(spec, f, indent=2)

main_chart
#https://altair-viz.github.io/user_guide/marks/image.html


alt.ConcatChart(...)